# JASA length estimation — Ablation 1 (Colab)

Run the strict leave-one-vehicle-out (LOVO) target ablation from the JASA weekly plan:

- shared proxy: `env_10db_width_x_speed_m` (envelope −10 dB width × speed)
- targets: overall length (`length_m`) versus wheelbase (`wheelbase_m`)
- comparison: fold-wise training-vehicle mean versus affine physics rule
- decision: prefer wheelbase when it beats its mean baseline and length does not

The notebook clones DopplerLab directly from GitHub, reads VS13 audio from Google Drive, persists extracted features/results to Drive, and can resume without recomputing features.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount("/content/drive")

# ---- Edit these paths if your Drive layout differs ----
GITHUB_REPO = "https://github.com/seetharamkkv/DopplerLab.git"
GIT_BRANCH = "main"
REPO_DIR = Path("/content/DopplerLab")

SHARED_ROOT = Path("/content/drive/Shareddrives/Spectral Transformers - Doppler/DopplerLab")
DATASET_ROOT = SHARED_ROOT / "datasets" / "vs13"
DRIVE_WORK_DIR = SHARED_ROOT / "length_estimation" / "jasa_ablation1"

# False reuses DRIVE_WORK_DIR/features.csv when it already exists.
FORCE_REEXTRACT = False
# -------------------------------------------------------

FEATURES_PATH = DRIVE_WORK_DIR / "features.csv"
RESULTS_DIR = DRIVE_WORK_DIR / "outputs" / "jasa_week"
DRIVE_WORK_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset: ", DATASET_ROOT)
print("Features:", FEATURES_PATH)
print("Results: ", RESULTS_DIR)
print("Reuse features:", FEATURES_PATH.is_file() and not FORCE_REEXTRACT)

## 1. Clone the current GitHub branch

Rerunning this cell refreshes the Colab checkout. Results and features remain safe because they are stored in Google Drive.

If the repository is private, leave `USE_GITHUB_PAT = True` and paste a classic PAT with `repo` scope when prompted. Public repos can set `USE_GITHUB_PAT = False`.

**Important:** `length_estimation/src/length_estimation/jasa_week.py` and `jasa-ablation1` must already be on the chosen `GIT_BRANCH`.

In [ ]:
import shutil
import subprocess
from getpass import getpass
from urllib.parse import quote, urlparse, urlunparse

# Set False for a public repo; True asks for a classic GitHub PAT (repo scope).
USE_GITHUB_PAT = True

os.chdir("/content")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_url = GITHUB_REPO
if USE_GITHUB_PAT:
    token = getpass("GitHub classic PAT (repo scope): ")
    parsed = urlparse(GITHUB_REPO)
    if parsed.scheme != "https":
        raise ValueError("GITHUB_REPO must be an https:// URL when USE_GITHUB_PAT=True")
    auth_netloc = f"{quote(token, safe='')}@{parsed.netloc}"
    clone_url = urlunparse(parsed._replace(netloc=auth_netloc))

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, clone_url, str(REPO_DIR)],
    check=True,
)
os.chdir(REPO_DIR)
print("branch:", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("HEAD:  ", subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

ablation_module = REPO_DIR / "length_estimation" / "src" / "length_estimation" / "jasa_week.py"
assert ablation_module.is_file(), (
    f"Missing {ablation_module}. Push the JASA ablation implementation to {GIT_BRANCH} first."
)
print("Clone OK:", REPO_DIR)

## 2. Install the length-estimation package

Colab already provides PyTorch. The repository requirements install the signal-processing, data, plotting, and editable package dependencies used by feature extraction and evaluation.

In [ ]:
requirements = REPO_DIR / "length_estimation" / "requirements.txt"
subprocess.run(
    ["python", "-m", "pip", "install", "-q", "-r", str(requirements)],
    check=True,
)

# Keep commands below independent of the notebook's current working directory.
LENGTH_ROOT = REPO_DIR / "length_estimation"
os.chdir(LENGTH_ROOT)
print("Installed from:", LENGTH_ROOT)

## 3. Validate VS13 and extract features

Expected Drive layout:

```text
vs13/
  Mazda3/
    Mazda3_50.wav
    Mazda3_50.txt
  ...
```

Each sidecar contains `speed_kmh cpa_time_s`. The vehicle names must match `length_estimation/data/vehicle_specs.csv` in the repository.

Ablation 1 needs the envelope proxy. Reassigned-STFT features are also retained so the same `features.csv` is ready for Ablation 2; synchrosqueezing is skipped because neither weekly ablation uses it.

In [ ]:
assert DATASET_ROOT.is_dir(), f"VS13 dataset directory not found: {DATASET_ROOT}"
wav_files = list(DATASET_ROOT.rglob("*.wav"))
txt_files = list(DATASET_ROOT.rglob("*.txt"))
print(f"Found {len(wav_files)} WAV files and {len(txt_files)} sidecars")
assert wav_files, "No WAV files found. Check DATASET_ROOT."

if FORCE_REEXTRACT or not FEATURES_PATH.is_file():
    cmd = [
        "python", "-m", "length_estimation.run",
        "--data-dir", str(DATASET_ROOT),
        "--output-dir", str(DRIVE_WORK_DIR),
        "features", "--no-ssq",
    ]
    print("Extracting features; this is the longest step...")
    subprocess.run(cmd, check=True, cwd=LENGTH_ROOT)
else:
    print("Reusing existing features:", FEATURES_PATH)

assert FEATURES_PATH.is_file(), f"Feature extraction did not create {FEATURES_PATH}"

## 4. Run Ablation 1

For every held-out vehicle, both affine calibration and the mean baseline are fitted using only the other 12 vehicles. All 400 clips are pooled only after each clip has received its out-of-vehicle prediction.

In [ ]:
cmd = [
    "python", "-m", "length_estimation.run",
    "--output-dir", str(DRIVE_WORK_DIR),
    "jasa-ablation1",
    "--features", str(FEATURES_PATH),
    "--jasa-output-dir", str(RESULTS_DIR),
]
subprocess.run(cmd, check=True, cwd=LENGTH_ROOT)

SUMMARY_PATH = RESULTS_DIR / "ablation1_summary.json"
TABLE_PATH = RESULTS_DIR / "ablation1_table.md"
assert SUMMARY_PATH.is_file(), f"Missing result: {SUMMARY_PATH}"
print("Saved all artifacts under:", RESULTS_DIR)

## 5. Inspect the LOVO decision

Expected decision from the weekly plan:

- length affine LOVO MAE ≈ **0.250 m** (fails vs mean ≈ 0.235 m)
- wheelbase affine LOVO MAE ≈ **0.096 m** (beats mean ≈ 0.105 m)
- winner → `wheelbase_m`

In [ ]:
import json
from IPython.display import Markdown, display
import pandas as pd

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
display(Markdown(TABLE_PATH.read_text(encoding="utf-8")))

print("Winning target:", summary["decision"]["winning_target"])
print("Rationale:     ", summary["decision"]["rationale"])
print("n_clips:       ", summary["n_clips"])
print("n_vehicles:    ", summary["n_vehicles"])

display(
    pd.DataFrame(
        {
            "target": ["length_m", "length_m", "wheelbase_m", "wheelbase_m"],
            "model": ["mean baseline", "affine env−10 dB×v", "mean baseline", "affine env−10 dB×v"],
            "lovo_mae_m": [
                summary["length_m"]["mean_baseline"]["lovo_mae"],
                summary["length_m"]["affine_env10db"]["lovo_mae"],
                summary["wheelbase_m"]["mean_baseline"]["lovo_mae"],
                summary["wheelbase_m"]["affine_env10db"]["lovo_mae"],
            ],
            "beats_mean": [
                "—",
                summary["length_m"]["beats_mean"],
                "—",
                summary["wheelbase_m"]["beats_mean"],
            ],
        }
    )
)

print("Artifact files:")
for path in sorted(RESULTS_DIR.glob("ablation1_*")):
    print(" -", path.name)

## Optional: continue to Ablation 2 + final \(W_b\to L\) rule

After Ablation 1 locks the target to wheelbase, the same Drive feature cache can run the remainder of the weekly funnel. Skip this cell if you only need the Ablation 1 decision.

In [ ]:
# Set True when you want the full weekly funnel on this same features.csv
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    cmd = [
        "python", "-m", "length_estimation.run",
        "--output-dir", str(DRIVE_WORK_DIR),
        "jasa-week",
        "--features", str(FEATURES_PATH),
        "--jasa-output-dir", str(RESULTS_DIR),
    ]
    subprocess.run(cmd, check=True, cwd=LENGTH_ROOT)
    final_summary = RESULTS_DIR / "final_rule_summary.json"
    assert final_summary.is_file(), f"Missing {final_summary}"
    display(json.loads(final_summary.read_text(encoding="utf-8")))
else:
    print("Skipped full pipeline. Set RUN_FULL_PIPELINE = True to run Ablation 2 + final rule.")